# FLUX 3 Image Klein on SageMaker (ml.g7e.2xlarge): standalone test

Black Forest Labs' [FLUX 3 Image Klein](https://huggingface.co/black-forest-labs/flux-3-image-klein) (5.6B, step-distilled,
16 steps) with its [prompt upsampler](https://huggingface.co/black-forest-labs/flux-3-image-klein-pup) (Qwen3.5-9B
fine-tune), both in BF16 and fully resident on one NVIDIA RTX PRO 6000 Blackwell (96 GB). One endpoint does
text-to-image, editing, multi-reference composition (up to 10 references) and bounding-box layouts.

**Stack**

| Layer | Choice | Why |
|---|---|---|
| Container | AWS **Ray Serve DLC for SageMaker** `ray:serve-ml-sagemaker-cuda-v1.5.7` | The AWS-maintained successor of the PyTorch inference DLC (that line stopped at PyTorch 2.6 / CUDA 12.4, which has no Blackwell kernels). Ships PyTorch 2.10 (CUDA 12.8 wheels, sm_120), CUDA 12.9.1, Transformers 5.19, Python 3.13, and adapts `/ping` + `/invocations` to Ray Serve. No custom image to build. Hugging Face's SageMaker DLCs have no CUDA 12.8+ build. |
| Host AMI | `al2023-ami-sagemaker-inference-gpu-4-1` | NVIDIA driver 580 / CUDA 13.0; the Ray DLC needs an explicit inference AMI on GPU. |
| Pipeline | Diffusers **0.42.1** `ModularPipeline` (`Flux3DistilledModularPipeline`) | Hugging Face's day-one FLUX 3 integration, same blocks as BFL's reference code; first release with FLUX 3. |
| Upsampler | Transformers `Qwen3_5ForConditionalGeneration` + **flash-linear-attention 0.6.0** | Greedy decoding with BFL's presence penalty (1.5 for text-to-image); Triton kernels for the Qwen3.5 linear-attention layers instead of the slow torch fallback. Triton needs a C compiler the image lacks, so the app installs `gcc` from the Amazon Linux repos at startup (~15 s). |
| Precision | BF16 everywhere, VAE in FP32 | ~39 GB of weights on a 96 GB GPU: no CPU offload, no quantization. |

**Flow:** stage the weights to S3 once (a CPU Processing job, ~10 min) → upload the serving code (seconds) → deploy
(~10-15 min, retried when g7e capacity is short) → generate → **delete the endpoint**.

This notebook tests the model on its own, with its own endpoint (`flux3-klein-g7e`) and I/O prefix. The AI Photo Booth
uses the same serving code through `../FLUX2-Klein-sagemaker/deploy.sh`, which creates the booth's
`FluxAiBoot-flux3-g7e` config and reuses weights this notebook has already staged in the same bucket.

**License.** FLUX 3 Image Klein is under the [FLUX Kommunity License](https://huggingface.co/black-forest-labs/flux-3-image-klein/blob/main/LICENSE.md):
non-commercial use, plus commercial use of outputs only for organizations under US$5M revenue. Testing and evaluation in a
non-production environment is a Non-Commercial Purpose; anything with end users (e.g. a public photo booth) needs a
license from BFL. Accept the license on the model page with the Hugging Face account whose token you use below.

## 1. Setup

In [ ]:
# boto3 must know ProductionVariant.InferenceAmiVersion (added in late 2024). Pillow is for the demos.
%pip install -q --upgrade "boto3>=1.40" pillow

In [ ]:
import os, sys, json, time, importlib
from pathlib import Path

import boto3

sys.path.insert(0, str(Path.cwd()))
import flux3_sm as fx
importlib.reload(fx)  # pick up edits to flux3_sm.py without restarting the kernel

# ============================== configuration ==============================
REGION        = boto3.session.Session().region_name or "us-east-1"
ACCOUNT       = boto3.client("sts").get_caller_identity()["Account"]
BUCKET        = f"sagemaker-{REGION}-{ACCOUNT}"     # any bucket in REGION the role can read/write
ENDPOINT_NAME = "flux3-klein-g7e"
INSTANCE_TYPE = "ml.g7e.2xlarge"                     # 1x RTX PRO 6000 Blackwell 96 GB, 8 vCPU, 64 GiB
HF_TOKEN      = os.environ.get("HF_TOKEN", "")      # <-- or paste a read token (license accepted on the repo)
AUTO_STOP_HOURS = 3                                  # server-side safety net; None to disable
REUSE_ENDPOINT  = True                               # keep a healthy endpoint on re-runs; False after editing serving/
# ===========================================================================

def _role() -> str:
    if os.environ.get("SAGEMAKER_ROLE_ARN"):
        return os.environ["SAGEMAKER_ROLE_ARN"]
    try:
        import sagemaker
        return sagemaker.get_execution_role()
    except Exception as exc:
        raise RuntimeError("Not running in SageMaker: set SAGEMAKER_ROLE_ARN to an execution role ARN") from exc

cfg = fx.Settings(region=REGION, bucket=BUCKET, role_arn=_role(),
                  endpoint_name=ENDPOINT_NAME, instance_type=INSTANCE_TYPE)
print("region   ", cfg.region)
print("role     ", cfg.role_arn)
print("image    ", cfg.image_uri)
print("model    ", cfg.model_uri, "(-> /opt/ml/model)")
print("endpoint ", cfg.endpoint_name, "on", cfg.instance_type)

### Preflight
The endpoint quota for `ml.g7e.2xlarge` is separate from the notebook/Studio quota of the same name.

In [ ]:
sq = cfg.client("service-quotas")
quota = next((q for page in sq.get_paginator("list_service_quotas").paginate(ServiceCode="sagemaker")
              for q in page["Quotas"] if q["QuotaName"] == f"{INSTANCE_TYPE} for endpoint usage"), None)
print(f"{INSTANCE_TYPE} for endpoint usage:", quota["Value"] if quota else "not found")
assert quota and quota["Value"] >= 1, "request a quota increase for the endpoint instance type first"

cfg.client("s3").head_bucket(Bucket=BUCKET)
existing = [e["EndpointName"] for e in cfg.client("sagemaker").list_endpoints(NameContains=ENDPOINT_NAME)["Endpoints"]]
print("bucket ok; existing endpoints with this name:", existing or "none")

## 2. Stage the weights to S3 (once)

`scripts/stage_weights.py` reads the repo's own `modular_model_index.json`, so it stages exactly the components the
pipeline loads: `transformer/` (11.1 GB), `text_encoder/` (Qwen3-VL-4B, 8.9 GB), `vae/` (0.3 GB), and the prompt
upsampler from its own repo into `prompt_upsampler/` (18.8 GB), ~39 GB in all. It skips the single-file package and the
FP8/NVFP4 variants. It runs as a SageMaker Processing job on a CPU instance with the same Ray DLC image, so the transfer
stays inside AWS. The Hugging Face token is put in a temporary Secrets Manager secret (`AmazonSageMaker-*` names are
readable by `AmazonSageMakerFullAccess`) and deleted as soon as the job ends; it never appears in the job's description
and is not needed again. Re-running is cheap: completed files are skipped, and the cell does nothing once
`_flux3_stage.json` exists.

In [ ]:
manifest = fx.weights_staged(cfg)
if manifest:
    print(f"already staged: {manifest['repo']}@{manifest['revision'][:12]}, "
          f"{manifest['total_bytes'] / 1e9:.1f} GB, {len(manifest['files'])} objects, at {manifest['staged_at']}")
else:
    assert HF_TOKEN, "set HF_TOKEN in the configuration cell (the repo is gated)"
    secret = fx.create_hf_token_secret(cfg, HF_TOKEN)
    try:
        job = fx.stage_weights_job(cfg, hf_token_secret=secret)   # ~10 min on ml.c6i.4xlarge
    finally:
        fx.delete_secret(cfg, secret)                            # the token is not kept in AWS
    manifest = fx.weights_staged(cfg)
    print(f"staged {manifest['total_bytes'] / 1e9:.1f} GB in {len(manifest['files'])} objects")
print(json.dumps(manifest["components"], indent=1))

## 3. Upload the serving code

Two small objects next to the weights. The DLC entrypoint installs `code/requirements.txt` (Diffusers 0.42.1,
accelerate, flash-linear-attention; the image's torch/transformers are pinned as guards) and runs
`serve run /opt/ml/model/config.yaml`. Re-run this cell and section 4 after editing `serving/`; the weights never move.

In [ ]:
fx.upload_serving(cfg)
print(open("serving/config.yaml").read())

## 4. Deploy

`fx.deploy` deletes any endpoint with this name first, creates model → endpoint config → endpoint, and waits.
If SageMaker reports no g7e capacity it backs off (2, 5, 10, 15 min) and tries again, at least 3 times; any other
failure stops with SageMaker's `FailureReason`. A capacity failure takes ~30 minutes to surface, so the whole cell
can take a while when g7e is scarce. With `REUSE_ENDPOINT = True`, an endpoint of this name that is already InService
with the same image and model data is kept (set it to `False` after editing `serving/`, so the new code loads).

The Ray DLC's `/ping` goes healthy as soon as Ray Serve has a route, which is *before* the 39 GB load finishes, so
`InService` is followed by `fx.wait_ready`, which waits for the app's own `[flux3] READY` line in CloudWatch (it
prints the load milestones on the way). Requests sent in between simply wait in the queue.

In [ ]:
t0 = time.time()
desc = fx.deploy(cfg, retries=3, reuse=REUSE_ENDPOINT)
if AUTO_STOP_HOURS:
    fx.set_auto_stop(cfg, hours=AUTO_STOP_HOURS)
fx.wait_ready(cfg)
print(f"ready in {(time.time() - t0) / 60:.1f} min")

## 5. Generate

Requests are JSON (see `serving/code/flux3_app.py` for every field). `response_format: "json"` returns the image plus
the caption the upsampler wrote, the seed and timings; the default `"png"` returns raw PNG bytes, like the FLUX.2
booth endpoint. Results land in `s3://<bucket>/flux3-klein-g7e/io/outputs/`.

In [ ]:
from IPython.display import display
from PIL import Image

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

def show(result, name, width=512):
    image = result.image
    image.save(OUT / f"{name}.png")
    if result.meta:  # response_format="json"
        t = result.meta["timings_s"]
        print(f"{name}: {image.width}x{image.height}, seed {result.meta['seed']}, {result.seconds:.1f}s end to end "
              f"(upsample {t['upsample']}s, generate {t['generate']}s)")
    else:            # raw PNG: no metadata travels with it
        print(f"{name}: {image.width}x{image.height} PNG, {result.seconds:.1f}s end to end")
    if result.caption and result.meta.get("upsampled"):
        print("caption:", result.caption[:700] + ("..." if len(result.caption) > 700 else ""))
    display(image.resize((width, round(width * image.height / image.width)), Image.LANCZOS))

### 5.1 Text-to-image: a short request, expanded by the upsampler

In [ ]:
t2i = fx.invoke(cfg, {
    "prompt": "A snow leopard on a moss-covered rock at dawn, golden backlight through pine needles",
    "seed": 42, "response_format": "json",
})
show(t2i, "t2i")

### 5.2 Editing: one reference image

In [ ]:
edit = fx.invoke(cfg, {
    "prompt": "turn it into a thick impasto oil painting with a warm golden palette",
    "images": [fx.encode_image(t2i.image)],
    "seed": 7, "response_format": "json",
})
show(edit, "edit")

### 5.3 Your own photo
Put an image next to the notebook (PNG/JPEG/WebP). A single reference keeps its size (downscaled only above 4 MP).

In [ ]:
LOCAL_IMAGE = "my_image.jpg"
EDIT = "replace the background with a neon-lit city street at night, keep the people exactly as they are"
if Path(LOCAL_IMAGE).exists():
    mine = fx.invoke(cfg, {"prompt": EDIT, "images": [fx.encode_image(LOCAL_IMAGE)], "seed": 0,
                           "response_format": "json"})
    show(mine, "my_edit")
else:
    print(f"{LOCAL_IMAGE} not found; skipping")

### 5.4 Multi-reference composition
Refer to references as "the first image", "the second image", ... (up to 10).

In [ ]:
multi = fx.invoke(cfg, {
    "prompt": "the snow leopard from the first image, painted in the style of the second image, "
              "sitting in an autumn forest at golden hour",
    "images": [fx.encode_image(t2i.image), fx.encode_image(edit.image)],
    "height": 768, "width": 1360, "seed": 123, "response_format": "json",
})
show(multi, "multi", width=680)

### 5.5 Layout with bounding boxes
`bbox` is `[y_min, x_min, y_max, x_max]` in 0-1. The upsampler writes the caption around the boxes.

In [ ]:
layout = fx.invoke(cfg, {
    "prompt": "hot-air balloons over a green valley at sunrise",
    "boxes": [
        {"id": "balloon_1", "desc": "a red and yellow striped hot-air balloon", "bbox": [0.08, 0.06, 0.48, 0.38]},
        {"id": "balloon_2", "desc": "a small blue hot-air balloon", "bbox": [0.20, 0.62, 0.42, 0.80]},
    ],
    "seed": 5, "response_format": "json",
})
show(layout, "layout")

### 5.6 Your own caption, no upsampler, PNG response
`upsample: false` sends the prompt as written: use it for full scene captions (yours or a cached upsampler caption).
Without `response_format` the endpoint returns PNG bytes, the contract the FLUX.2 booth already speaks.

In [ ]:
raw = fx.invoke(cfg, {"prompt": t2i.caption, "upsample": False, "seed": 43, "height": 1024, "width": 1024})
show(raw, "caption_reuse_png")

## 6. Latency on this instance
`upsample` and `generate` are measured inside the container; `end to end` adds the S3 round trips and the async
queue. The same numbers are in CloudWatch as `[flux3] GENERATED` lines.

In [ ]:
runs = {"t2i": t2i, "edit": edit, "multi": multi, "layout": layout}
if "mine" in globals():
    runs["my_edit"] = mine
print(f"{'request':10} {'size':>10} {'refs':>4} {'upsample':>9} {'generate':>9} {'end to end':>11} {'peak GPU':>9}")
for name, r in runs.items():
    t = r.meta["timings_s"]
    print(f"{name:10} {r.meta['width']:>5}x{r.meta['height']:<4} {r.meta['references']:>4} {t['upsample']:>8.1f}s "
          f"{t['generate']:>8.1f}s {r.seconds:>10.1f}s {r.meta.get('gpu_peak_gb') or 0:>7.1f}GB")
print(f"{'png':10} {raw.image.width:>5}x{raw.image.height:<4} {0:>4} {'-':>9} {'-':>9} {raw.seconds:>10.1f}s")

## 7. Clean up

**Delete the endpoint when you are done**: an `ml.g7e.2xlarge` is billed for every hour it exists. The weights
(~39 GB in S3) stay, so the next deploy skips staging. Auto-stop (section 4) scales an idle endpoint to 0 instances
at the scheduled time as a safety net; it does not delete it.

In [ ]:
fx.delete(cfg)

In [ ]:
# Optional: also remove the staged weights (the next deploy would then have to stage them again).
# boto3.resource("s3").Bucket(BUCKET).objects.filter(Prefix=cfg.project + "/").delete()